In [ ]:
import operator
import os
import re
from groq import Groq
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph,START,END,MessagesState
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field

In [ ]:
class FraudState(MessagesState):
    job_text:str
    suspicious_keyword:Annotated[list[str],operator.add]
    risk_score:int
    tool_call_iteration:int
    emailaddress:list[str]
    is_email_free:bool

    website_link:list[str]
    website_available:bool
    website_content_score:int       #score between 0-40 based on corporate info, address, and career listings.
    valide_website:bool

    linkedin_score: int             #score between 0-20 based on employee count and official presence.
    Linkedin_profile:bool
    evidance_log:Annotated[list[str],operator.add]



In [ ]:
class SuspiciousPhraseExtractor(BaseModel):
    """Schema to force the llm to output a clean list of suspicious phrases"""
    phrases:list[str]=Field(description="List of exact phrases extracted from the text that indicate upfront payments, unrealistic salary promises, or pressure tactics."
    )

class ContentVerification(BaseModel):
    is_legitimate_content: bool = Field(description="True if website content matches official corporate operations.")
    content_score: int = Field(description="Score between 0-40 based on corporate info, address, and career listings.")
    reasoning: str = Field(description="Explanation of how the content was evaluated and why it was deemed legitimate or not.")
    is_valid_profile: bool = Field(description="True if an official LinkedIn company page exists.")
    linkedin_score: int = Field(description="Score between 0-20 based on employee count and official presence.")    


In [ ]:
@tool
def job_post_search_tool(company_name:str,website_link:str)->TavilySearchResults:
    """Searches Tavily to verify if a hiring company has a real website 
    or registered online footprint in Bangladesh.And they are actively hiring for the position mentioned in the job circular.
    
    Args:
        company_name: Name of the hiring company extracted from the job circular.
        website_link: Optional website link found in the post (e.g., 'companybd.com').
    """
    if website_link:
        search_query=f"{company_name}OR{website_link} official site Bangladesh and recent job postings"
    else:
        search_query=f"{company_name} official site Bangladesh and recent job postings"
    try:
        search_results=TavilySearchResults.search(search_query,max_results=3)
        summary = f"Search Findings for query [{search_query}]:\n"
        for idx,result in enumerate(search_results.results):
            tile=result.get("title","No title")
            url=result.get("url","No URL")
            snippet=result.get("content","No content")
            summary+=f"Result {idx+1}:\nTitle: {tile}\nURL: {url}\nSnippet: {snippet}\n\n"
        return summary
    except Exception as e:
        return f"Search failed for '{search_query}'. Error details: {str(e)}"


In [ ]:
@tool
def linkedin_search_tool(company_name:str)->TavilySearchResults:
    """Searches Tavily to verify if a hiring company has an official LinkedIn page.
    
    Args:
        company_name: Name of the hiring company extracted from the job circular.
    """
    search_query=f"{company_name} official LinkedIn page"
    try:
        search_results=TavilySearchResults.search(search_query,max_results=3)
        summary = f"LinkedIn Search Findings for query [{search_query}]:\n"
        for idx,result in enumerate(search_results.results):
            tile=result.get("title","No title")
            url=result.get("url","No URL")
            snippet=result.get("content","No content")
            summary+=f"Result {idx+1}:\nTitle: {tile}\nURL: {url}\nSnippet: {snippet}\n\n"
        return summary
    except Exception as e:
        return f"LinkedIn search failed for '{search_query}'. Error details: {str(e)}"

In [ ]:
tools = [job_post_search_tool, linkedin_search_tool]
tool_node = ToolNode(tools)

In [ ]:
llm=Groq.LLM(api_key=os.environ.get("Groq_API_KEY"),model="qwen/qwen3.8-27b",temperature=0,max_tokens=512)
structured_llm=llm.structured_output(SuspiciousPhraseExtractor)
structured_llm_content_verification=llm.structured_output(ContentVerification)
llm_with_tools=llm.with_tools(tools)

In [ ]:
def input_normalizer_node(state:FraudState):
    
    email_pattern=r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    raw_email=list(re.findall(email_pattern,state["job_text"]))
    free_email_domains=["gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com"]
    for email in raw_email:
        domain=email.split("@")[-1].lower()
        if domain in free_email_domains:
            state["is_email_free"]=True
        else:
            state["is_email_free"]=False

    website_pattern=r'https?://(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
    raw_website=list(re.findall(website_pattern,state["job_text"]))  
    if len(raw_website)>0:
        website_available=True
    else:
        website_available=False
              
    job_text=state["job_text"].strip()
    return{
        "job_text":job_text,
        "emailaddress":raw_email,
        "is_email_free":state["is_email_free"],
        "website_link":raw_website,
        "website_available":website_available,
        "evidance_log":[f"Step 1: Extracted emails: {raw_email}, Free email: {state['is_email_free']}, Extracted websites: {raw_website}, Website available: {website_available}"]
    }

NameError: name 'FraudState' is not defined

In [ ]:
def key_word_extractor(state:FraudState):
    """This node extracts suspicious phrases from the job text using the structured LLM."""
    job_text=state["job_text"]
    response:SuspiciousPhraseExtractor=structured_llm(f"You are a recruitment fraud analyst in Bangladesh. Extract any phrases "
        "from this job circular that request money upfront (e.g., bKash, registration fee), "
        "offer unrealistic pay for basic skills, or pressure the applicant to act quickly:\n\n"
        f"{job_text}\n\n" "if there are no such phrases,return an empty list.")
    extracted_phrases=response.phrases
    return{
        "suspicious_keyword":extracted_phrases,
        "evidance_log":[f"Step:2 Extracted suspicious phrases: {extracted_phrases}"]
    }
    

In [ ]:
def risk_base_on_keywords_node(state:FraudState):
    """this node investigates the keyword risk and email addresses risk.
    """
    risk_score = 0
    if state["is_email_free"]:
        risk_score+=5
    if not state["website_available"]:
        risk_score+=5

    if len(state["suspicious_keyword"])>5:
        risk_score+=30
    elif(len(state["suspicious_keyword"])<=5 and len(state["suspicious_keyword"])>0):
        risk_score+=len(state["suspicious_keyword"])*6
    else:
        risk_score+=0
    return{
        "risk_score":risk_score
        }
    

In [ ]:
def investigator_node(state:FraudState):
    system_prompt=(
        "You are a recruitment fraud investigator for JobShield BD.\n"
        "RULES:\n"
        "1. You MUST verify BOTH the company's official website AND their LinkedIn presence.\n"
        "2. If you have only checked one of them, you MUST invoke a search tool for the missing one.\n"
        "3. Only stop calling tools when you have gathered clear search evidence for BOTH website and LinkedIn."
    )
    current_iteration = state.get("tool_call_iteration", 0)
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    result=llm_with_tools(messages)
    new_log_entry=f"Step{current_iteration+3}: Investigation findings:{result.content}"
    return {"messages": [result],
            "tool_call_iteration":current_iteration+1,
            "evidance_log": state.get("evidance_log", []) + [new_log_entry]
            }

In [ ]:
def score_formatter_node(state: FraudState):
    """Extracts BOTH website and LinkedIn scores into the unified Pydantic schema."""
    final_report = structured_llm.invoke(state["messages"])
    risk_score=state["risk_score"]+state["website_content_score"]+state["linkedin_score"]
    return {
        "risk_score": risk_score
        }

In [ ]:
def router_next(state:FraudState):
    """Determines the next node based on the evaluation results."""
    if state["tool_call_iteration"] <= 3:
        return "investigator_node"
    last_message=state["messages"][-1]
    if hasattr(last_message,"tool_calls") and last_message.tool_calls:
        return "investigator_node"
  
    return "score_formatter_node"

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Dict, Any

def verdict_node(state: FraudState) -> Dict[str, Any]:
    system_prompt = (
        "You are the Lead Fraud Analyst for JobShield BD. Your job is to render a final verdict "
        "on whether a recruitment posting or company is legitimate or fraudulent based strictly "
        "on the search evidence provided.\n\n"
        "OUTPUT FORMAT REQUIREMENTS:\n"
        "1. VERDICT: State clearly [LEGITIMATE, SUSPICIOUS, or FRAUDULENT].\n"
        "2. RISK SCORE: Assign a risk score from 0 to 100.\n"
        "3. WEBSITE EVIDENCE: Summarize findings regarding their official website.\n"
        "4. LINKEDIN EVIDENCE: Summarize findings regarding their LinkedIn presence.\n"
        "5. KEY RED FLAGS / CONFIRMATIONS: Bullet point specific evidence leading to this decision.\n"
        "6. ACTIONABLE RECOMMENDATION: Final advice for the job seeker in Bangladesh."
    )

    # Extract conversation history or formatted search evidence
    formatted_evidence = state.get("evidance_log", "")
    
    # Fallback to last message content if formatted_evidence is empty
    if not formatted_evidence and state["messages"]:
        formatted_evidence = state["messages"][-1].content

    user_prompt = f"Analyze the following gathered evidence and issue your final verdict:\n\n{formatted_evidence}"

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]

    # Call the LLM (without tools) to generate the final decision
    response = llm.invoke(messages)

    # Store final verdict and append the final AI response to state
    return {
        "messages": [response],
        "final_verdict": response.content
    }

Graph Assembly